In [0]:

-- Create Silver streaming table for cleaned and validated data
CREATE OR REFRESH STREAMING TABLE first_data_engineering_project.silver.silver_customers

-- Define data quality expectations for the silver table
(
    CONSTRAINT valid_customer_id
    EXPECT (customer_id IS NOT NULL AND customer_id > 0)
    ON VIOLATION DROP ROW,

    CONSTRAINT valid_city 
    EXPECT (city IS NOT NULL) ON VIOLATION DROP ROW,

    CONSTRAINT valid_signup_date 
    EXPECT (signup_date IS NOT NULL AND signup_date <= CURRENT_DATE()) ON VIOLATION DROP ROW
)

-- Describe the purpose of the Silver table
COMMENT "Cleaned and validated customer table"

-- Define Silver table properties
TBLPROPERTIES(
    "quality" = "silver",
    "pipelines.reset.allowed" = false
)

AS

-- Deduplicate and keep the most recently ingested record

WITH deduplicate AS(
    SELECT *,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY ingestion_timestamp DESC) AS rn
    FROM STREAM first_data_engineering_project.bronze.bronze_customers
)

SELECT
    customer_id,
    city,
    signup_date,
    CURRENT_TIMESTAMP AS ingestion_timestamp
FROM deduplicate
WHERE rn = 1
;